<a href="https://colab.research.google.com/github/ArtemFilin1990/Dadatatele2/blob/main/Bearing_Catalog_Data_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re

def parse_bearing_designation(designation):
    """
    Разделяет обозначение подшипника на префикс, базовый номер и суффикс.
    """
    if pd.isna(designation) or not str(designation).strip():
        return "", "", ""
    d = str(designation).strip()

    # Регулярное выражение: ищет префикс (буквы/дефисы), затем первую группу цифр (базовый номер), затем остаток (суффикс)
    match = re.search(r'^([A-Za-z\s\-]+)?(\d+)(.*)$', d)
    if match:
        pref = match.group(1).strip() if match.group(1) else ""
        num = match.group(2).strip()
        suf = match.group(3).strip() if match.group(3) else ""
        return pref, num, suf
    return "", d, ""

def process_catalog(input_dir="."):
    print("Начинаем процесс сборки каталогов...")

    # 1. Загрузка справочников (Источники истины)
    ref_gost_iso_files = glob.glob(os.path.join(input_dir, "*REF_GOST_ISO_размеры*.csv"))
    ref_iso_suf_files = glob.glob(os.path.join(input_dir, "*REF_ISO_суффиксы*.csv"))

    ref_gost_iso = pd.read_csv(ref_gost_iso_files[0], dtype=str) if ref_gost_iso_files else pd.DataFrame()
    ref_iso_suffixes = pd.read_csv(ref_iso_suf_files[0], dtype=str) if ref_iso_suf_files else pd.DataFrame()

    # Построение карты соответствий ГОСТ <-> ISO
    gost_iso_map = {}
    if not ref_gost_iso.empty:
        for _, row in ref_gost_iso.iterrows():
            gost = str(row.get('Отечественный', '')).strip()
            iso = str(row.get('Импортный', '')).strip()
            if not gost or not iso or gost == 'nan' or iso == 'nan': continue

            d = pd.to_numeric(row.get('d', np.nan), errors='coerce')
            D = pd.to_numeric(row.get('D', np.nan), errors='coerce')
            B = pd.to_numeric(row.get('B', np.nan), errors='coerce')
            m = pd.to_numeric(row.get('Масса', np.nan), errors='coerce')

            # Двунаправленный поиск для связок
            gost_iso_map[gost] = {'analog': iso, 'd': d, 'D': D, 'B': B, 'm': m}
            gost_iso_map[iso] = {'analog': gost, 'd': d, 'D': D, 'B': B, 'm': m}

    # 2. Загрузка брендовых CSV-файлов
    all_csvs = glob.glob(os.path.join(input_dir, "*_enriched.xlsx - *.csv")) + \
               glob.glob(os.path.join(input_dir, "*_patch.xlsx - *.csv"))

    ignore_keywords = ["SCHEMA", "MASTER_PRODUCTS", "DICT_FERSA_CODES", "CROSSREF_TPI", "RAW_TPI_SHAFTS", "REF_GOST_ISO", "REF_ISO"]
    brand_files = [f for f in all_csvs if not any(k in f for k in ignore_keywords)]

    df_list = []
    for f in brand_files:
        try:
            df = pd.read_csv(f, dtype=str)
            df_list.append(df)
        except Exception as e:
            print(f"Ошибка чтения {f}: {e}")

    if not df_list:
        print("ОШИБКА: Не найдены файлы брендов для обработки.")
        return

    df_all = pd.concat(df_list, ignore_index=True)

    # 3. Жёсткие правила очистки
    df_all = df_all.dropna(subset=['номер'])
    df_all = df_all[df_all['номер'].str.strip() != '']

    # Логика определения ГОСТ/ISO
    gost_brand_markers = ['ГПЗ', 'СВПЗ', 'МПЗ', 'ЕПК', 'КНС', '10-ГПЗ']

    def detect_gost(row):
        brand = str(row.get('Бренд', '')).upper()
        if any(g in brand for g in gost_brand_markers):
            return True
        analog_raw = str(row.get('Аналог', ''))
        if '[ISO]' in analog_raw:  # Если аналог помечен как ISO, оригинал — это ГОСТ
            return True
        return False

    df_all['is_gost'] = df_all.apply(detect_gost, axis=1)

    out_rows_gost = []
    out_rows_iso = []

    for _, row in df_all.iterrows():
        brand = str(row.get('Бренд', ''))
        prod = str(row.get('продукция', 'Подшипник')).strip()
        if prod.lower() == 'nan' or not prod:
            prod = 'Подшипник'

        pref = str(row.get('префикс', '')).replace('nan', '').strip()
        num = str(row.get('номер', '')).replace('nan', '').strip()
        suf = str(row.get('суффикс', '')).replace('nan', '').strip()

        full_orig = f"{pref}{num}{suf}"

        d = pd.to_numeric(row.get('d_mm'), errors='coerce')
        D = pd.to_numeric(row.get('D_mm'), errors='coerce')
        B = pd.to_numeric(row.get('B_mm'), errors='coerce')
        mass = pd.to_numeric(row.get('mass_kg'), errors='coerce')

        analog_raw = str(row.get('Аналог', '')).replace('nan', '').strip()
        direct_analog = "NO DIRECT EQUIV"

        # Извлечение чистого обозначения аналога из APROM/ISO мусорных тегов
        if "[ISO]" in analog_raw:
            direct_analog = analog_raw.split("[ISO]")[-1].strip()
        elif "[APROM]" in analog_raw:
            direct_analog = analog_raw.split("[APROM]")[-1].strip()
        elif analog_raw:
            direct_analog = analog_raw

        # Правило: Сначала ищи прямую пару в REF_GOST_ISO_размеры
        if full_orig in gost_iso_map:
            mapped = gost_iso_map[full_orig]
            direct_analog = mapped['analog']
            if pd.isna(d): d = mapped['d']
            if pd.isna(D): D = mapped['D']
            if pd.isna(B): B = mapped['B']
            if pd.isna(mass): mass = mapped['m']
        else:
            # Правило 6: Совпадение одного номера без подтверждения размеров -> NO DIRECT EQUIV
            if not direct_analog or direct_analog == full_orig:
                direct_analog = "NO DIRECT EQUIV"
            elif pd.isna(d) or pd.isna(D):
                direct_analog = "NO DIRECT EQUIV"

        # Разбор аналога на части, если он существует
        if direct_analog != "NO DIRECT EQUIV":
            a_pref, a_num, a_suf = parse_bearing_designation(direct_analog)
        else:
            a_pref, a_num, a_suf = "", "", ""

        out_row = {
            'Бренд': brand,
            'продукция': prod,
            'префикс': pref,
            'номер': num,
            'суффикс': suf,
            'префикс аналога': a_pref,
            'номер аналога': a_num,
            'суффикс аналога': a_suf,
            'Аналог': direct_analog,
            'd мм': d,
            'D мм': D,
            'B мм': B,
            'M кг': mass
        }

        if row['is_gost']:
            out_rows_gost.append(out_row)
        else:
            out_rows_iso.append(out_row)

    df_gost = pd.DataFrame(out_rows_gost)
    df_iso = pd.DataFrame(out_rows_iso)

    # 4. Дедупликация
    dedup_cols = ['Бренд', 'префикс', 'номер', 'суффикс', 'Аналог']
    df_gost = df_gost.drop_duplicates(subset=dedup_cols)
    df_iso = df_iso.drop_duplicates(subset=dedup_cols)

    # 5. Лист SCHEMA
    schema_data = [
        {"field": "Бренд", "type": "text", "required": "yes", "description": "Бренд источника"},
        {"field": "продукция", "type": "text", "required": "yes", "description": "Тип продукции"},
        {"field": "префикс", "type": "text", "required": "no", "description": "Часть обозначения до базового номера"},
        {"field": "номер", "type": "text", "required": "yes", "description": "Базовый номер подшипника"},
        {"field": "суффикс", "type": "text", "required": "no", "description": "Часть обозначения после базового номера"},
        {"field": "префикс аналога", "type": "text", "required": "no", "description": "Префикс обозначения аналога"},
        {"field": "номер аналога", "type": "text", "required": "no", "description": "Базовый номер аналога"},
        {"field": "суффикс аналога", "type": "text", "required": "no", "description": "Суффикс обозначения аналога"},
        {"field": "Аналог", "type": "text", "required": "no", "description": "Полное обозначение прямого аналога"},
        {"field": "d мм", "type": "number", "required": "no", "description": "Внутренний диаметр"},
        {"field": "D мм", "type": "number", "required": "no", "description": "Наружный диаметр"},
        {"field": "B мм", "type": "number", "required": "no", "description": "Ширина"},
        {"field": "M кг", "type": "number", "required": "no", "description": "Масса"}
    ]
    df_schema = pd.DataFrame(schema_data)

    # 6. Листы PREFIX_DICT и SUFFIX_DICT
    all_prefixes = pd.concat([df_gost['префикс'], df_iso['префикс']]).replace('', np.nan).dropna().unique()
    all_suffixes = pd.concat([df_gost['суффикс'], df_iso['суффикс']]).replace('', np.nan).dropna().unique()

    df_pref = pd.DataFrame([{'Код': p, 'Расшифровка': ''} for p in all_prefixes])

    suf_rows = []
    for s in all_suffixes:
        desc = ""
        if not ref_iso_suffixes.empty and 'Код' in ref_iso_suffixes.columns:
            match = ref_iso_suffixes[ref_iso_suffixes['Код'] == s]
            if not match.empty:
                desc = match['Расшифровка'].iloc[0]
        suf_rows.append({'Код': s, 'Расшифровка': desc})
    df_suf = pd.DataFrame(suf_rows)

    # 7. Экспорт в Excel (с соблюдением порядка колонок)
    cols_order = [s["field"] for s in schema_data]
    df_gost = df_gost[cols_order]
    df_iso = df_iso[cols_order]

    out_file = "Compiled_Bearing_Catalog.xlsx"
    with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
        df_gost.to_excel(writer, sheet_name="GOST", index=False)
        df_iso.to_excel(writer, sheet_name="ISO", index=False)
        df_schema.to_excel(writer, sheet_name="SCHEMA", index=False)
        df_pref.to_excel(writer, sheet_name="PREFIX_DICT", index=False)
        df_suf.to_excel(writer, sheet_name="SUFFIX_DICT", index=False)

    print("=== ОТЧЕТ О СБОРКЕ ===")
    print(f"Строк в GOST: {len(df_gost)}")
    print(f"Строк в ISO: {len(df_iso)}")
    print(f"Кодов в PREFIX_DICT: {len(df_pref)}")
    print(f"Кодов в SUFFIX_DICT: {len(df_suf)}")

    no_equiv_gost = len(df_gost[df_gost['Аналог'] == 'NO DIRECT EQUIV'])
    no_equiv_iso = len(df_iso[df_iso['Аналог'] == 'NO DIRECT EQUIV'])
    print(f"Строк получили NO DIRECT EQUIV (всего): {no_equiv_gost + no_equiv_iso}")
    print(f"Файл успешно сохранен как {out_file}")

if __name__ == "__main__":
    process_catalog()

Начинаем процесс сборки каталогов...
ОШИБКА: Не найдены файлы брендов для обработки.
